# WL-AKSVD Molecular Classification & Interpretability Notebook

**End-to-end pipeline**: SMILES string → molecular graph → WL hashing → AKSVD sparse coding → classification → node-level interpretability → RDKit visualization with subtree highlighting.

## Architecture

```
SMILES  ──►  RDKit Mol  ──►  NetworkX Graph  ──►  WL Hashing  ──►  AKSVD Sparse Code  ──►  Classifier
                                                       │                    │                    │
                                                  token–node map     dictionary atoms       coef_ weights
                                                       └────────────────────┴────────────────────┘
                                                                    │
                                                          Node-level attribution
                                                                    │
                                                          RDKit atom highlighting
```

## 1. Imports & Setup

In [ ]:
import sys
import warnings
import numpy as np
import networkx as nx
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# RDKit
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, Descriptors
from rdkit.Chem.Draw import rdMolDraw2D

# Visualisation
from IPython.display import display, Image as IPImage
from PIL import Image
import io

# Pipeline modules (adjust path if needed)
# sys.path.insert(0, "/path/to/your/project")
from graph_encoders.wl import WL
from dict_learners.aksvd import AKSVD
from utils.graph_data import GraphDataLoader
from interpreter.wl_aksvd_interpreter import WLAKSVDInterpreter

warnings.filterwarnings("ignore")

## 2. Inspect NCI Graph Format

Before building the SMILES converter, we need to know **exactly** what node
attributes the NCI dataset uses.  `WeisfeilerLehmanHashing(attributed=True)`
reads the `"feature"` key from each node.  This cell prints a sample graph
so you can verify the attribute name and value format.

In [ ]:
data_loader = GraphDataLoader()
graphs, y = data_loader.nci_full_graphs, data_loader.nci_full_labels

print(f"Dataset size : {len(graphs)} graphs")
print(f"Label dist.  : {Counter(y)}")
print()

# Inspect first graph
sample_g = graphs[0]
print(f"Sample graph : {sample_g.number_of_nodes()} nodes, {sample_g.number_of_edges()} edges")
print(f"Node attrs   : {dict(list(sample_g.nodes(data=True))[:5])}")
print(f"Edge attrs   : {dict(list(sample_g.edges(data=True))[:3])}")

# Determine the attribute key and value type the pipeline expects
sample_node = list(sample_g.nodes(data=True))[0]
print(f"\nNode 0 raw   : id={sample_node[0]}, attrs={sample_node[1]}")
print(f"Attr keys    : {list(sample_node[1].keys())}")

## 3. SMILES → NetworkX Graph Converter

This is the **critical bridge** between RDKit molecules and the existing WL
pipeline.  The converter must produce a NetworkX graph whose node attributes
are byte-for-byte identical to the NCI training graphs so that the learned WL
vocabulary can match incoming tokens.

> **Action required**: after running the inspection cell above, verify that
> `ATTR_KEY` and the value format in `_atom_feature()` match your NCI data.
> The default below assumes nodes carry `feature = str(atomic_number)`, which
> is the standard TU-dataset / karateclub convention.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Configuration — adjust these if your NCI graphs use a different format
# ──────────────────────────────────────────────────────────────────────────
ATTR_KEY = "feature"          # node attribute key read by WeisfeilerLehmanHashing


def _atom_feature(atom) -> str:
    """
    Convert an RDKit atom to the same string representation used in the NCI
    training graphs.

    Default: atomic number as a plain string ("6" for carbon, "7" for nitrogen).
    Change this function if your NCI graphs use element symbols, one-hot
    vectors, or a different encoding.
    """
    return str(atom.GetAtomicNum())


def smiles_to_graph(smiles: str) -> nx.Graph:
    """
    Parse a SMILES string and return a NetworkX graph compatible with the
    WL-AKSVD pipeline.

    Node attributes match the NCI training format so that WL hashing
    produces tokens from the same vocabulary.

    Parameters
    ----------
    smiles : str
        A valid SMILES string (e.g. "c1ccccc1" for benzene).

    Returns
    -------
    nx.Graph
        Undirected graph with consecutive integer node IDs (0-indexed)
        and the expected node attribute.

    Raises
    ------
    ValueError
        If RDKit cannot parse the SMILES string.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"RDKit could not parse SMILES: '{smiles}'")

    # Add explicit hydrogens if the NCI graphs include them
    # (uncomment the next line if your NCI data includes H atoms)
    # mol = Chem.AddHs(mol)

    g = nx.Graph()

    for atom in mol.GetAtoms():
        g.add_node(atom.GetIdx(), **{ATTR_KEY: _atom_feature(atom)})

    for bond in mol.GetBonds():
        g.add_edge(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())

    return g


def smiles_to_mol_and_graph(smiles: str):
    """Return both the RDKit Mol and the pipeline-ready NetworkX graph."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"RDKit could not parse SMILES: '{smiles}'")
    graph = smiles_to_graph(smiles)
    return mol, graph

### Quick sanity check — compare a converted SMILES graph with an NCI graph

In [ ]:
# Convert benzene and compare attribute format with NCI
test_g = smiles_to_graph("c1ccccc1")
nci_sample = graphs[0]

print("SMILES graph (benzene):")
print(f"  Nodes: {dict(list(test_g.nodes(data=True))[:3])}")
print()
print("NCI sample graph:")
print(f"  Nodes: {dict(list(nci_sample.nodes(data=True))[:3])}")
print()

smiles_attrs = set(list(test_g.nodes(data=True))[0][1].keys())
nci_attrs   = set(list(nci_sample.nodes(data=True))[0][1].keys())
if smiles_attrs == nci_attrs:
    print("✅ Attribute keys match — converter is compatible.")
else:
    print(f"⚠️  Mismatch: SMILES uses {smiles_attrs}, NCI uses {nci_attrs}")
    print("   Update ATTR_KEY and _atom_feature() to match.")

## 4. Train the Full Pipeline

Standard three-split strategy:
- **Vocab split** — learns the WL vocabulary (discriminative feature selection)
- **ML split** — trains the classifier on AKSVD sparse codes
- **Test split** — held-out evaluation

This cell also stores the fitted components needed by the interpreter.

In [ ]:
# ── Data splits ───────────────────────────────────────────────────────────
G_train, G_test, y_train, y_test = train_test_split(
    graphs, y, test_size=0.2, random_state=42
)
G_vocab_train, G_ML_train, y_vocab_train, y_ML_train = train_test_split(
    G_train, y_train, test_size=0.75, random_state=42
)

print(f"Vocab train : {len(G_vocab_train)}")
print(f"ML train    : {len(G_ML_train)}")
print(f"Test        : {len(G_test)}")

# ── WL hashing ────────────────────────────────────────────────────────────
wl = WL()
wl_embeddings_vocab = wl.generate_training_embeddings(G_vocab_train, y_vocab_train)
print(f"Vocab size  : {wl.n_vocab}")

# ── AKSVD dictionary learning ────────────────────────────────────────────
aksvd = AKSVD()
aksvd.fit(training_graph_embeddings=wl_embeddings_vocab)
print(f"Dictionary  : {aksvd._dictionary.shape}")

# ── Generate sparse codes for ML train & test ─────────────────────────────
wl_emb_ml_train = wl.generate_inferencing_embeddings(G_ML_train)
X_ML_train = aksvd.infer(wl_emb_ml_train)

wl_emb_test = wl.generate_inferencing_embeddings(G_test)
X_ML_test = aksvd.infer(wl_emb_test)

# ── Scaling ───────────────────────────────────────────────────────────────
scaler = MaxAbsScaler()
X_ML_train_scaled = scaler.fit_transform(X_ML_train)
X_ML_test_scaled = scaler.transform(X_ML_test)

### Train Logistic Regression for interpretability

In [ ]:
# Logistic Regression is preferred for interpretability because coef_
# gives per-atom signed attribution (unlike tree-based feature_importances_)
classifier = LogisticRegression(
    max_iter=1000,
    solver="saga",
    random_state=42,
    class_weight="balanced",
)
classifier.fit(X_ML_train_scaled, y_ML_train)

y_pred = classifier.predict(X_ML_test_scaled)
print(classification_report(y_test, y_pred))
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")

## 5. Set Up the Interpreter

In [ ]:
LABEL_MAP = {0: "Non-Cancerous", 1: "Cancerous"}

interpreter = WLAKSVDInterpreter(
    wl=wl,
    aksvd=aksvd,
    classifier=classifier,
    scaler=scaler,
    training_graphs=G_ML_train,
    training_labels=y_ML_train,
    training_sparse_codes=X_ML_train,   # unscaled, for cosine similarity
    label_map=LABEL_MAP,
)

print("✅ Interpreter ready.")

## 6. Visualisation Engine

The functions below take the interpreter's node-importance output and render
a publication-quality RDKit image with:

- **Atom highlighting** — colour intensity proportional to attribution magnitude
- **Colour semantics** — red atoms push *toward* the predicted class, blue atoms
  push *against* it (or vice-versa, customisable)
- **Annotation** — importance percentages on each atom

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors


def _importance_colour_map(
    node_importance: dict,
    sorted_nodes: list,
    supporting_cmap: str = "Reds",
    opposing_cmap: str = "Blues",
    floor_alpha: float = 0.15,
) -> dict:
    """
    Map each node to an RGBA tuple based on signed importance.

    Supporting nodes  → red gradient   (pushes toward predicted class)
    Opposing nodes    → blue gradient  (pushes against predicted class)

    Nodes with zero importance get a faint wash so they remain visible.
    """
    if not sorted_nodes:
        return {}

    max_abs = max(abs(s) for _, s in sorted_nodes) or 1.0
    sup_cm = cm.get_cmap(supporting_cmap)
    opp_cm = cm.get_cmap(opposing_cmap)

    colours = {}
    for node_id, score in sorted_nodes:
        intensity = abs(score) / max_abs
        # Scale into [floor_alpha, 1.0] so even low-importance atoms are visible
        mapped = floor_alpha + intensity * (1.0 - floor_alpha)

        if score >= 0:
            rgba = sup_cm(mapped)
        else:
            rgba = opp_cm(mapped)

        colours[node_id] = rgba

    return colours


def visualise_prediction(
    smiles: str,
    interpreter: "WLAKSVDInterpreter",
    top_k_atoms: int = 5,
    img_size: tuple = (600, 400),
    show_atom_indices: bool = True,
    show_percentages: bool = True,
    supporting_cmap: str = "Reds",
    opposing_cmap: str = "Blues",
) -> Image.Image:
    """
    End-to-end: SMILES → classification → node-importance → highlighted image.

    Parameters
    ----------
    smiles           : SMILES string to classify and explain.
    interpreter      : fitted WLAKSVDInterpreter.
    top_k_atoms      : number of dictionary atoms to trace.
    img_size         : (width, height) of the output image.
    show_atom_indices: annotate atoms with their index.
    show_percentages : annotate atoms with importance %.
    supporting_cmap  : matplotlib colourmap name for supporting atoms.
    opposing_cmap    : matplotlib colourmap name for opposing atoms.

    Returns
    -------
    PIL.Image.Image  — the rendered molecule with highlighting.
    """
    mol, graph = smiles_to_mol_and_graph(smiles)

    # Run interpretability
    result = interpreter.get_node_importance(graph, top_k_atoms=top_k_atoms)

    # Compute 2D coordinates for a clean layout
    AllChem.Compute2DCoords(mol)

    # Build colour map
    sorted_nodes = result["sorted_nodes"]
    colour_map = _importance_colour_map(
        result["node_importance"],
        sorted_nodes,
        supporting_cmap=supporting_cmap,
        opposing_cmap=opposing_cmap,
    )

    # Prepare RDKit highlight arguments
    highlight_atoms = list(colour_map.keys())
    highlight_atom_colours = {
        int(k): tuple(v[:3]) for k, v in colour_map.items()
    }

    # Highlight bonds connecting two highlighted atoms
    highlight_bonds = []
    highlight_bond_colours = {}
    for bond in mol.GetBonds():
        a1, a2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        if a1 in colour_map and a2 in colour_map:
            bid = bond.GetIdx()
            highlight_bonds.append(bid)
            # Average the colours of the two endpoint atoms
            c1 = np.array(colour_map[a1][:3])
            c2 = np.array(colour_map[a2][:3])
            highlight_bond_colours[bid] = tuple((c1 + c2) / 2)

    # Draw with the SVG renderer for crisp output
    drawer = rdMolDraw2D.MolDraw2DCairo(*img_size)
    opts = drawer.drawOptions()
    opts.useBWAtomPalette()

    if show_atom_indices:
        for atom in mol.GetAtoms():
            idx = atom.GetIdx()
            if show_percentages and idx in result["node_importance_pct"]:
                pct = result["node_importance_pct"][idx]
                atom.SetProp("atomNote", f"{pct:.1f}%")
            else:
                atom.SetProp("atomNote", str(idx))

    drawer.DrawMolecule(
        mol,
        highlightAtoms=highlight_atoms,
        highlightAtomColors=highlight_atom_colours,
        highlightBonds=highlight_bonds,
        highlightBondColors=highlight_bond_colours,
    )
    drawer.FinishDrawing()

    bio = io.BytesIO(drawer.GetDrawingText())
    return Image.open(bio)


def full_analysis(
    smiles: str,
    interpreter: "WLAKSVDInterpreter",
    top_k_atoms: int = 5,
    img_size: tuple = (700, 450),
):
    """
    Full end-to-end analysis: classification, text report, and visualisation.
    """
    mol, graph = smiles_to_mol_and_graph(smiles)

    # 1. Full text report
    report = interpreter.full_report(
        graph,
        top_k_atoms=top_k_atoms,
        top_k_features_per_atom=5,
        top_k_similar=5,
    )
    print(report)

    # 2. Highlighted visualisation
    img = visualise_prediction(
        smiles,
        interpreter,
        top_k_atoms=top_k_atoms,
        img_size=img_size,
    )
    display(img)

    return img

## 7. Classify & Visualise a Compound

Enter any valid SMILES string below.  The pipeline will:

1. Convert it to a molecular graph
2. Run WL hashing → AKSVD sparse coding → Logistic Regression
3. Trace the prediction back to individual atoms
4. Render the molecule with importance highlighting

**Colour legend**:
- 🔴 **Red** atoms *support* the predicted class (push the prediction toward it)
- 🔵 **Blue** atoms *oppose* the predicted class (push toward the other class)
- Intensity ∝ attribution magnitude
- Percentages on each atom show share of total attribution

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# ⬇️  CHANGE THIS SMILES STRING TO CLASSIFY ANY MOLECULE  ⬇️
# ──────────────────────────────────────────────────────────────────────────
SMILES = "c1ccc2c(c1)cc1ccc3cccc4ccc2c1c34"   # Pyrene

img = full_analysis(SMILES, interpreter, top_k_atoms=5)

## 8. Batch Analysis — Compare Multiple Compounds

In [ ]:
COMPOUNDS = {
    "Benzene":          "c1ccccc1",
    "Naphthalene":      "c1ccc2ccccc2c1",
    "Phenol":           "Oc1ccccc1",
    "Aniline":          "Nc1ccccc1",
    "Cyclohexane":      "C1CCCCC1",
    "Aspirin":          "CC(=O)Oc1ccccc1C(=O)O",
}

results_summary = []

for name, smi in COMPOUNDS.items():
    mol, graph = smiles_to_mol_and_graph(smi)
    explanation = interpreter.explain_prediction(graph, top_k_atoms=5)

    results_summary.append({
        "Compound": name,
        "SMILES": smi,
        "Prediction": explanation["prediction_label"],
        "Confidence": (
            f"{explanation['confidence']*100:.1f}%"
            if explanation["confidence"] is not None else "N/A"
        ),
        "Active Atoms": explanation["n_active_atoms"],
    })

    print(f"{'─'*50}")
    print(f"  {name} ({smi})")
    print(f"  → {explanation['prediction_label']}"
          f"  (conf: {explanation['confidence']*100:.1f}%)"
          if explanation['confidence'] else "")
    print(f"  Active dictionary atoms: {explanation['n_active_atoms']}")

    img = visualise_prediction(smi, interpreter, top_k_atoms=5, img_size=(400, 300))
    display(img)

print(f"\n{'─'*50}")
print("Summary:")
for r in results_summary:
    print(f"  {r['Compound']:<16} → {r['Prediction']:<16}  conf={r['Confidence']}")

## 9. Side-by-Side Comparison

In [ ]:
def side_by_side(smiles_list, names=None, interpreter=interpreter, img_size=(350, 250)):
    """
    Render multiple molecules side-by-side with importance highlighting.
    """
    if names is None:
        names = [f"Mol {i}" for i in range(len(smiles_list))]

    images = []
    for smi in smiles_list:
        img = visualise_prediction(smi, interpreter, img_size=img_size)
        images.append(img)

    # Stitch images horizontally
    total_w = sum(im.width for im in images) + 10 * (len(images) - 1)
    max_h = max(im.height for im in images) + 30  # room for label
    canvas = Image.new("RGB", (total_w, max_h), "white")

    x_offset = 0
    for im, name in zip(images, names):
        canvas.paste(im, (x_offset, 0))
        x_offset += im.width + 10

    display(canvas)
    return canvas


# Example comparison
side_by_side(
    ["c1ccccc1", "c1ccc2ccccc2c1", "CC(=O)Oc1ccccc1C(=O)O"],
    names=["Benzene", "Naphthalene", "Aspirin"],
)

## 10. Validation — Run Interpreter on NCI Test Graphs

Confirm that the interpreter produces consistent results on graphs from the
original NCI dataset (not SMILES-derived).

In [ ]:
# Pick 3 random test graphs
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(G_test), size=3, replace=False)

for i in sample_idx:
    print(f"\n{'═'*60}")
    print(f"  Test graph index: {i}")
    print(f"  True label: {LABEL_MAP.get(y_test[i], y_test[i])}")
    print(f"{'═'*60}")

    explanation = interpreter.explain_prediction(G_test[i], top_k_atoms=5)
    breakdown = interpreter.contribution_breakdown(explanation)
    print(breakdown)

    node_report = interpreter.node_highlight_report(G_test[i], top_k_atoms=5, top_n_nodes=8)
    print(node_report)

## Notes & Troubleshooting

**Format mismatch**:  If the SMILES converter produces WL tokens that don't
overlap with the learned vocabulary, the sparse code will be all zeros and the
prediction will be arbitrary.  Check:

1. Run cell 2 to inspect `feature` attribute values on NCI nodes.
2. Confirm `_atom_feature()` returns the same type/format.
3. If NCI uses element symbols (`"C"`, `"N"`) instead of atomic numbers (`"6"`, `"7"`),
   change `_atom_feature` to `return atom.GetSymbol()`.

**Hydrogen atoms**:  NCI graphs typically omit hydrogens (heavy-atom-only).
The default converter does the same.  If your NCI data includes H, uncomment
the `Chem.AddHs(mol)` line in `smiles_to_graph()`.

**Alternative classifiers**: The interpreter supports `RandomForestClassifier`
and `GradientBoostingClassifier` via `feature_importances_`, but these give
unsigned (magnitude-only) attribution — direction information is lost.
`LogisticRegression` with `coef_` is strongly recommended for full signed
attribution.